In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from py2neo import Graph

sys.path.append('..')

from graph_utilities import create_relationships, create_nodes, ensure_unique_constraint

In [ ]:
load_dotenv()

NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
DATA_DIR = os.getenv("DEMO_DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
IDS = ['subject_id', 'hadm_id', "icd_code", "icd_version", "itemid"]

# Load data from csv

In [ ]:
patients = pd.read_csv(os.path.join(DATA_DIR,"patients.csv.gz"))
admissions = pd.read_csv(os.path.join(DATA_DIR,"admissions.csv.gz"))
diagnoses = pd.read_csv(os.path.join(DATA_DIR,"diagnoses_icd.csv.gz"))
d_icd = pd.read_csv(os.path.join(DATA_DIR,"d_icd_diagnoses.csv.gz"))
labevents = pd.read_csv(os.path.join(DATA_DIR,"labevents.csv.gz"))
microbiologyevents = pd.read_csv(os.path.join(DATA_DIR,"microbiologyevents.csv.gz"))
d_labitems = pd.read_csv(os.path.join(DATA_DIR,"d_labitems.csv.gz"))

# Take a look at data

In [ ]:
patients.head()

In [ ]:
admissions.head()

In [ ]:
diagnoses.head()

In [ ]:
d_icd.head()

In [ ]:
labevents.head()

In [ ]:
diagnoses.groupby('subject_id').size().describe()

In [ ]:
patients.shape, admissions.shape, diagnoses.shape

# Data selection

In [ ]:

patients_small = patients[["subject_id", "anchor_age", "gender", "anchor_year"]].drop_duplicates()
admissions_small = admissions[[
    "hadm_id","subject_id","admittime", "deathtime", "admission_type", 'admission_location', 
    'discharge_location', 'insurance', 'language', 'marital_status', 'race']].drop_duplicates()
diagnoses_small = diagnoses[["hadm_id","subject_id", "icd_code", "icd_version"]].drop_duplicates()

# patients_small = patients_small.head(1)
admissions_small = admissions_small[admissions_small['subject_id'].isin(patients_small['subject_id'])]
diagnoses_small = diagnoses_small[diagnoses_small['subject_id'].isin(patients_small['subject_id'])]

d_icd_small = d_icd.merge(
    diagnoses_small[['icd_code', 'icd_version']].drop_duplicates(),
    on=['icd_code', 'icd_version'],
    how='inner'
)

labevents_small = labevents.merge(
    admissions_small[['subject_id', 'hadm_id']].drop_duplicates(),
    on=['subject_id', 'hadm_id'],
    how='inner'
)
d_labitems_small = d_labitems.merge(
    labevents_small[['itemid']].drop_duplicates(),
    on=['itemid'],
    how='inner'
)

In [ ]:
patients_small.shape, admissions_small.shape, diagnoses_small.shape

# Graph generation in Neo4j

In [ ]:
graph = Graph(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD), name='test2')
graph.delete_all()

In [ ]:
node_type_ids = {
    "Patient": ["subject_id"],
    "Admission": ["hadm_id"],
    "Diagnosis": ["icd_code", "icd_version"],
    "LabItem": ["itemid"],
}

In [ ]:
for node_type, id_fields in node_type_ids.items():
    ensure_unique_constraint(graph, node_type, id_fields)

## Nodes

In [ ]:
patient_nodes = create_nodes(
    graph=graph, 
    df=patients_small, 
    node_type="Patient", 
    id_fields=["subject_id"], 
    name_field="subject_id", 
    exclude_attrs=IDS,
)

admission_nodes = create_nodes(
    graph=graph, 
    df=admissions_small, 
    node_type="Admission", 
    id_fields=["hadm_id"], 
    name_field="hadm_id", 
    exclude_attrs=IDS,
)

diagnosis_nodes = create_nodes(
    graph=graph, 
    df=d_icd_small, 
    node_type="Diagnosis", 
    id_fields=["icd_version", "icd_code"], 
    name_field="long_title", 
    exclude_attrs=IDS,
)

labitems_nodes = create_nodes(
    graph=graph, 
    df=d_labitems_small, 
    node_type="LabItem", 
    id_fields=["itemid"], 
    name_field="label", 
    exclude_attrs=IDS,
)

## Edges

In [ ]:
create_relationships(
    graph=graph,
    df=admissions_small,
    node1_type="Patient",
    node2_type="Admission",
    node1_id_fields=["subject_id"],
    node2_id_fields=["hadm_id"],
    r_type="HAS_ADMISSION",
    exclude_fields=IDS,
)
print(f"HAS_ADMISSION relationships created")

create_relationships(
    graph=graph,
    df=diagnoses_small,
    node1_type="Admission",
    node2_type="Diagnosis",
    node1_id_fields=["hadm_id"],
    node2_id_fields=["icd_code", "icd_version"],
    r_type="HAS_DIAGNOSIS",
    exclude_fields=IDS,
)
print(f"HAS_DIAGNOSIS relationships created")

create_relationships(
    graph=graph,
    df=labevents_small,
    node1_type="Admission",
    node2_type="LabItem",
    node1_id_fields=["hadm_id"],
    node2_id_fields=["itemid"],
    r_type="HAS_LABEVENT",
    exclude_fields=IDS,
)
print(f"HAS_LABEVENT relationships created")